# Hiring Funnels, Noisy Interviews and Calibration

Companion notebook for the [Hiring and Team Building](https://codemia.io/courses/staff_and_engineering_manager_interviews/stf_hiring_and_team_building) and [How Level Decisions Are Made](https://codemia.io/courses/staff_and_engineering_manager_interviews/stf_how_level_decisions_are_made) lessons on Codemia.

Manager interviews often ask you to reason about hiring with numbers: how many candidates you need to source to make five hires, why one interviewer's opinion is not enough and why calibration across interviewers matters. In this notebook you will:

1. Size a hiring funnel and see which stage dominates the effort.
2. Simulate noisy interviewers and measure false hires and missed hires as the panel grows.
3. Add lenient and harsh interviewers, then fix them with calibration.
4. See how a strong staff candidate can be down leveled by noise alone.

Everything is a small simulation with stated assumptions, so change the numbers and rerun.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

## 1. Size the funnel

A funnel is a chain of conversion rates. Working backwards from the hires you need tells you how many candidates each stage must handle, and therefore how many interview hours the team will spend. The rates below are illustrative for a senior engineering role; replace them with your own.

In [ ]:
stages = [
    ("sourced",        1.00),
    ("recruiter screen", 0.35),   # share of sourced who pass to screen and pass it
    ("hiring manager", 0.50),
    ("onsite loop",    0.45),
    ("offer",          0.40),
    ("accepted",       0.75),
]

def funnel(hires, stages):
    overall = np.prod([r for _, r in stages])
    need = hires / overall
    counts = []
    n = need
    for name, r in stages:
        n = n * r
        counts.append((name, n))
    return need, counts

need, counts = funnel(5, stages)
print(f"to make 5 hires, source about {need:.0f} candidates")
for name, n in counts:
    print(f"  {name:18s} {n:7.1f}")

Now attach interview hours to each stage. The loop is where the team's time goes, so improving the **hiring manager screen** (fewer weak candidates reach the loop) often saves more hours than sourcing more people.

In [ ]:
hours = {"recruiter screen": 0.0, "hiring manager": 1.0, "onsite loop": 5 * 1.5, "offer": 1.0}

def team_hours(stages, hires=5):
    _, counts = funnel(hires, stages)
    entering = {name: counts[i - 1][1] for i, (name, _) in enumerate(counts) if i > 0}
    return sum(entering[s] * h for s, h in hours.items())

base = team_hours(stages)
better_screen = [(n, 0.35 if n == "hiring manager" else (0.60 if n == "onsite loop" else r)) for n, r in stages]
print(f"team hours, baseline:           {base:6.1f}")
print(f"team hours, stricter HM screen: {team_hours(better_screen):6.1f}")
assert team_hours(better_screen) < base

## 2. Interviews are noisy measurements

Model each candidate's true ability as a number, with the hiring bar at `1.0`. Each interviewer sees the true ability plus noise. The panel hires when the **average** score clears the bar. A **false hire** is someone below the bar who gets an offer; a **missed hire** is someone above the bar who does not.

In [ ]:
def simulate(panel, n=50_000, noise=0.8, bias=None, bar=1.0, calibrate=False):
    skill = rng.normal(0.0, 1.0, n)
    b = np.zeros(panel) if bias is None else np.asarray(bias[:panel])
    scores = skill[:, None] + b[None, :] + rng.normal(0.0, noise, (n, panel))
    if calibrate:
        scores = scores - b[None, :]          # perfect knowledge of each interviewer's lean
    hire = scores.mean(axis=1) > bar
    good = skill > bar
    false_hire = (hire & ~good).sum() / max(hire.sum(), 1)
    missed = (~hire & good).sum() / max(good.sum(), 1)
    return false_hire, missed

for p in [1, 2, 3, 4, 5, 6]:
    fh, ms = simulate(p)
    print(f"panel {p}: share of offers that are false hires {fh:5.1%}, strong candidates missed {ms:5.1%}")

In [ ]:
panels = range(1, 9)
res = [simulate(p) for p in panels]
plt.figure(figsize=(7, 4))
plt.plot(list(panels), [r[0] for r in res], marker="o", label="false hires among offers")
plt.plot(list(panels), [r[1] for r in res], marker="o", label="strong candidates missed")
plt.xlabel("interviewers in the panel")
plt.ylabel("rate")
plt.title("More independent signal, fewer mistakes, with diminishing returns")
plt.legend(); plt.grid(alpha=.3); plt.show()

Errors fall quickly from one to four interviewers and then flatten. That is one reason loops settle at four to six rounds: each extra round costs the same hours but buys less certainty. It also shows why **independent** feedback, written before the debrief, matters. If interviewers anchor on each other, five rounds behave like far fewer.

## 3. Lenient and harsh interviewers, then calibration

Real interviewers lean. Below, the pool runs from lenient (+1.0) to harsh (-1.0). With a random panel drawn from a pool, the leans add noise between candidates: who you happen to meet changes your outcome. Calibration, through shared rubrics, shadowing and reviewing each interviewer's history, removes much of that lean.

In [ ]:
pool = np.array([1.0, -1.0, 0.6, -0.6, 0.0, 0.8, -0.9, 0.2])

def simulate_pool(panel, calibrate, n=50_000, noise=0.8, bar=1.0):
    skill = rng.normal(0.0, 1.0, n)
    picks = np.array([rng.choice(len(pool), panel, replace=False) for _ in range(n)])
    lean = pool[picks]
    scores = skill[:, None] + lean + rng.normal(0.0, noise, (n, panel))
    if calibrate:
        scores = scores - lean
    hire = scores.mean(axis=1) > bar
    good = skill > bar
    return (hire & ~good).sum() / hire.sum(), (~hire & good).sum() / good.sum()

raw = simulate_pool(4, False)
cal = simulate_pool(4, True)
print(f"uncalibrated: false hires {raw[0]:5.1%}, missed {raw[1]:5.1%}")
print(f"calibrated:   false hires {cal[0]:5.1%}, missed {cal[1]:5.1%}")
assert cal[0] + cal[1] < raw[0] + raw[1]

Calibration does not make interviewers smarter; it removes a **systematic** error that more rounds cannot average away when the same lenient interviewer sits on many panels. In an interview, this is the numeric version of the answer: I review interviewer score distributions each quarter and pair outliers with calibrated interviewers.

## 4. Down leveling a real staff candidate

Now two bars: senior at `0.5` and staff at `1.2`. Take a candidate whose true ability is exactly at the staff bar plus a small margin, and ask how often a four person panel levels them at staff, senior or no hire. Compare an average rule with a stricter rule many committees use in practice: **no round may be below the senior bar** for a staff offer.

In [ ]:
def level_outcomes(true_skill, panel=4, noise=0.8, n=50_000, strict=False):
    scores = true_skill + rng.normal(0.0, noise, (n, panel))
    avg = scores.mean(axis=1)
    staff = avg > 1.2
    if strict:
        staff &= scores.min(axis=1) > 0.5
    senior = ~staff & (avg > 0.5)
    return staff.mean(), senior.mean(), 1 - staff.mean() - senior.mean()

for t in [1.3, 1.5, 1.8]:
    a = level_outcomes(t)
    s = level_outcomes(t, strict=True)
    print(f"true {t}: average rule staff {a[0]:5.1%} senior {a[1]:5.1%} | strict rule staff {s[0]:5.1%} senior {s[1]:5.1%}")

A candidate a little above the staff bar is down leveled a large share of the time by noise alone, and a "no weak round" rule makes it worse. Two practical lessons follow, both covered in the lessons:

- **For candidates**: the margin matters. Stories and designs that sit clearly above the bar survive noise; ones that just meet it often get leveled down.
- **For hiring managers**: when the packet is mixed, a targeted follow up round adds independent signal exactly where the uncertainty is, instead of a decision driven by one noisy score.

## Exercises

1. Change the onsite pass rate to `0.30` and the offer acceptance to `0.60`. How many candidates must you source for five hires, and how many loop hours does that cost?
2. In `simulate`, add correlation between interviewers (for example a shared random term per candidate that every interviewer sees). How does it change the value of adding rounds?
3. Add a follow up round only for candidates whose average lands within `0.2` of the staff bar. How much does it reduce down leveling, and what does it cost in extra rounds?